# Probability, and the question you are actually asking

MichAl Academy, lesson 1.8.

Run each cell with **Shift+Enter**.

Two halves. Conditional probability, which decides whether a detector is worth
switching on, and distributions, which decide whether your threshold means
anything.

## 1. The tree, in code

Ten thousand events. Split by what they really are, then by what the detector
said.

In [ ]:
TOTAL = 10_000


def tree(bad_per_10k, catches_pct, false_alarm_pct):
    bad  = bad_per_10k
    good = TOTAL - bad

    tp = round(bad  * catches_pct     / 100)   # bad, alerted
    fn = bad - tp                              # bad, missed
    fp = round(good * false_alarm_pct / 100)   # fine, alerted
    tn = good - fp                             # fine, quiet

    return {"tp": tp, "fn": fn, "fp": fp, "tn": tn}


t = tree(bad_per_10k=100, catches_pct=80, false_alarm_pct=1)
print(t)
print()
print("alerts raised:", t["tp"] + t["fp"])

In [ ]:
def precision(t):
    alerts = t["tp"] + t["fp"]
    return t["tp"] / alerts if alerts else float("nan")


def recall(t):
    real = t["tp"] + t["fn"]
    return t["tp"] / real if real else float("nan")


print(f"recall    {recall(t):6.1%}   <- fraction of real attacks caught")
print(f"precision {precision(t):6.1%}   <- fraction of alerts that were real")

A detector that catches 80% of attacks and misfires on only 1% of normal traffic
produces alerts that are wrong more often than they are right.

Both of those inputs sound like good numbers in a datasheet. Neither of them is
the number your analyst experiences.

## 2. Only one of the two moves with the base rate

In [ ]:
print(f"{'bad per 10k':>12} {'recall':>8} {'precision':>10} {'alerts':>8}")
for bad in [1000, 500, 200, 100, 50, 20, 10, 5]:
    t = tree(bad, catches_pct=80, false_alarm_pct=1)
    print(f"{bad:>12} {recall(t):>8.1%} {precision(t):>10.1%} {t['tp'] + t['fp']:>8}")

Recall does not move at all. It is a property of the detector.

Precision falls off a cliff. It is a property of the detector **and the world it
is pointed at**, and the world is the part the vendor did not test on.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

rates = np.arange(1, 501)
prec  = [precision(tree(int(b), 80, 1)) for b in rates]
rec   = [recall(tree(int(b), 80, 1))    for b in rates]

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(rates, prec, label="precision (alerts that are real)")
ax.plot(rates, rec,  label="recall (attacks caught)")
ax.set_xlabel("truly bad events per 10,000")
ax.set_ylabel("rate")
ax.set_ylim(0, 1)
ax.legend()
plt.show()

## 3. Bayes' rule is that fraction

What you just computed is Bayes' rule. Nothing was hidden.

```
P(bad | alert)  =  bad and alerted  /  (bad and alerted + fine and alerted)
```

The usual textbook form rearranges the same thing.

In [ ]:
def bayes(prior, sensitivity, false_alarm):
    """Textbook form: prior belief, updated by evidence."""
    numerator = sensitivity * prior
    evidence  = sensitivity * prior + false_alarm * (1 - prior)
    return numerator / evidence


by_counting = precision(tree(100, 80, 1))
by_formula  = bayes(prior=100 / TOTAL, sensitivity=0.80, false_alarm=0.01)

print(f"by counting the tree : {by_counting:.6f}")
print(f"by the formula       : {by_formula:.6f}")
print("same answer?", round(by_counting, 6) == round(by_formula, 6))

If a formula ever confuses you, draw the tree. Ten thousand is a good number to
split, because every leaf stays a whole number.

## 4. Distributions, and the three-sigma rule

Under a normal distribution about 99.7% of values sit within three standard
deviations of the mean. That single fact is where "alert on anything more than
three sigma from normal" comes from.

Security data mostly is not normal.

In [ ]:
rng = np.random.default_rng(42)

n = 20_000
normal      = rng.normal(500, 120, n)                 # a bell curve
heavy_tailed = rng.lognormal(mean=6.0, sigma=0.9, size=n)   # a long right tail

for name, col in [("normal", normal), ("heavy tailed", heavy_tailed)]:
    threshold = col.mean() + 3 * col.std()
    fired = (col > threshold).sum()
    print(f"{name:14s} mean {col.mean():8.1f}  sd {col.std():8.1f}  "
          f"3-sigma at {threshold:9.1f}  fires on {fired:5d} of {n} ({fired/n:.2%})")

Same rule, same sample size, wildly different alert volume. The heavy-tailed
column has a long right tail of perfectly ordinary large values, and the
three-sigma line sits inside it.

Look at the shapes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))

for ax, (name, col) in zip(axes, [("normal", normal), ("heavy tailed", heavy_tailed)]):
    ax.hist(col, bins=80)
    ax.axvline(col.mean() + 3 * col.std(), color="red", linestyle="--", label="mean + 3 sd")
    ax.set_title(name)
    ax.legend()

plt.tight_layout()
plt.show()

A quantile-quantile plot answers "is this normal" directly. Points on the line
mean normal, and a curve away from it at the top right is a heavy right tail.

In [ ]:
from scipy import stats

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
stats.probplot(normal, dist="norm", plot=axes[0])
axes[0].set_title("normal: points on the line")
stats.probplot(heavy_tailed, dist="norm", plot=axes[1])
axes[1].set_title("heavy tailed: tail peels off")
plt.tight_layout()
plt.show()

When the tail peels away like that, set the threshold from a **percentile of the
actual data** instead of a multiple of the standard deviation. You then choose
your alert volume directly, which is usually what you wanted anyway.

In [ ]:
for pct in [99.0, 99.5, 99.9]:
    cut = np.percentile(heavy_tailed, pct)
    fired = (heavy_tailed > cut).sum()
    print(f"{pct:5.1f}th percentile at {cut:9.1f}  fires on {fired:4d} of {n} ({fired/n:.2%})")

## 5. Your turn

A detector is being sold to you on these numbers: it catches 95% of attacks and
its false alarm rate is 2%.

Your environment sees 2,000,000 events a day, and about 40 of them are genuinely
malicious.

Work out how many alerts land on the analyst each day, and how many are real.

In [ ]:
EVENTS_PER_DAY = 2_000_000
REALLY_BAD     = 40
CATCHES        = 0.95
FALSE_ALARM    = 0.02

# TODO: replace the zeros
real_alerts  = 0
false_alerts = 0

total_alerts = real_alerts + false_alerts
print(f"alerts per day : {total_alerts:,}")
if total_alerts:
    print(f"of which real  : {real_alerts:,}  ({real_alerts / total_alerts:.3%})")
print()
print("did you fill it in?", total_alerts > 0)

The false alarm rate applies to the events that are **not** malicious, which is
almost all of them.

<details>
<summary>Answer</summary>

```python
real_alerts  = round(REALLY_BAD * CATCHES)                          # 38
false_alerts = round((EVENTS_PER_DAY - REALLY_BAD) * FALSE_ALARM)   # 39,999
```

38 real alerts arrive inside roughly 40,000. Under one alert in a thousand is
worth looking at, from a detector whose datasheet is not lying about anything.

That is not a detector problem. Nothing about the model is broken. It is that 2%
of two million is a very large number, and the thing being looked for is very
rare. Track 2 spends a whole lesson on what to do about it, and the answers are
mostly: raise the threshold and accept lower recall, add a second independent
signal, or reduce the population you run it on.

</details>

## What you now have

- `P(alert | bad)` and `P(bad | alert)` are different numbers, and vendors quote the first
- Draw the tree, split ten thousand, read the answer off the leaves
- Recall is a property of the detector, precision is a property of the detector and the world
- Bayes' rule is that fraction rearranged, nothing more
- Three sigma assumes a bell curve, and security data is mostly heavy-tailed
- Set thresholds from percentiles of your own data, and check with a QQ plot

Next is lesson 1.9: given two detectors and two numbers, how to tell a real
difference from noise.